In [1]:
"""
Tariff Analysis
This script compares price changes of products hit by Canada's March 2025 counter-tariffs
against products that weren't, using StatCan monthly retail prices.
"""
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

sys.path.append(str(Path.cwd().parent / "src"))
from db import get_engine

engine = get_engine()

In [2]:
# Monthly prices with each product's tariff status pulled from PostgreSQL db
query = """
SELECT p.date, p.product, p.price, c.status
FROM prices p
JOIN product_tariff_crosswalk c ON p.product = c.product
WHERE p.place = 'Canada'
ORDER BY p.product, p.date
"""
prices = pd.read_sql(text(query), engine, parse_dates=["date"])
print(prices.shape)
prices.head()

(12650, 4)


,date,product,price,status
0,2017-01-01,"Almonds, 200 grams",3.74,not tariffed
1,2017-02-01,"Almonds, 200 grams",4.45,not tariffed
2,2017-03-01,"Almonds, 200 grams",4.57,not tariffed
3,2017-04-01,"Almonds, 200 grams",4.42,not tariffed
4,2017-05-01,"Almonds, 200 grams",4.32,not tariffed


In [3]:
# Create price index - each product's price as a % of its Feb 2025 price (last month pre tariffs)
february_2025_prices = prices[prices["date"] == "2025-02-01"].set_index("product")["price"]

prices["price_index"] = prices["price"] / prices["product"].map(february_2025_prices) * 100
prices.head()

,date,product,price,status,price_index
0,2017-01-01,"Almonds, 200 grams",3.74,not tariffed,79.237288
1,2017-02-01,"Almonds, 200 grams",4.45,not tariffed,94.279661
2,2017-03-01,"Almonds, 200 grams",4.57,not tariffed,96.822034
3,2017-04-01,"Almonds, 200 grams",4.42,not tariffed,93.644068
4,2017-05-01,"Almonds, 200 grams",4.32,not tariffed,91.525424


In [ ]:
# Every product should be exactly 100 in Feb 2025
february_check = prices[prices["date"] == "2025-02-01"]["price_index"]
print(len(february_check), "products, index range:", february_check.min(), "to", february_check.max())

110 products, index range: 100.0 to 100.0


In [5]:
# Average price index per month for tariffed vs not tariffed groups
group_index = prices.groupby(["date", "status"])["price_index"].mean().unstack("status")
group_index.loc["2025-01-01":"2025-06-01"]

status,not tariffed,tariffed
date,,
2025-01-01,99.391255,99.763621
2025-02-01,100.000000,100.000000
2025-03-01,99.165968,99.973869
2025-04-01,98.847164,100.003990
2025-05-01,102.188276,101.925120
2025-06-01,101.997251,102.614831


In [ ]:
# Tariffed vs not-tariffed price index since 2023
import matplotlib.pyplot as plt

TARIFFED_COLOR = "#d1242f"       # red
NOT_TARIFFED_COLOR = "#1a73c2"   # blue

chart_data = group_index.loc["2023-01-01":]

fig, ax = plt.subplots(figsize=(10, 5))

# Tariff period - March 4 to August 31, 2025
ax.axvspan(pd.Timestamp("2025-03-01"), pd.Timestamp("2025-09-01"), color="grey", alpha=0.12, lw=0)

ax.plot(chart_data.index, chart_data["tariffed"], color=TARIFFED_COLOR, lw=2, label="Tariffed (39 products)")
ax.plot(chart_data.index, chart_data["not tariffed"], color=NOT_TARIFFED_COLOR, lw=2, label="Not tariffed (71 products)")
ax.axhline(100, color="grey", lw=0.8, ls="--")   # Feb 2025 starting point

# Label each line at its latest value
last_date = chart_data.index[-1]
for column, name in [("tariffed", "Tariffed"), ("not tariffed", "Not tariffed")]:
    last_value = chart_data[column].iloc[-1]
    ax.annotate(f"{name} {last_value:.0f}", xy=(last_date, last_value), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color="#333")

ax.text(pd.Timestamp("2025-06-01"), ax.get_ylim()[1], "Tariffs in force", ha="center", va="top", fontsize=9, color="#555")

ax.set_title("Grocery price index: tariffed vs not-tariffed products (Canada)", loc="left")
ax.set_ylabel("Price index (Feb 2025 = 100)")
ax.grid(axis="y", color="#e5e5e5", lw=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")

plt.tight_layout()
plt.show()